# RTENSO2026 2015/16 El Nino: anomaly maps and pattern ACC

Case-study companion to ESP-Lab `1a_*_leadtime_acc_skill_map.ipynb` and
`1b_*_leadtime_rmse_skill_map.ipynb`. With one target year, the ACC is the
**spatial** anomaly correlation (area-weighted, centred) between the forecast and
observed anomaly maps, computed per lead season, per member and for the ensemble
mean, over several regions; the RMSE is the matching area-weighted map error.

Anomalies remove the Reanalysis 1981-2010 lead-time climatology (lead-drift corrected
for Feb/Aug starts, see `CLIM_METHOD`) and the 1981-2010 observed monthly climatology. `TS` is verified
against HadISST2 over ice-free ocean (60S-60N) only; `TREFHT` and `PSL` against ERA5; `PRECT` against
GPCP v2.3.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from matplotlib.colors import BoundaryNorm
from cartopy.util import add_cyclic_point
from esp_lab.utils import colormap_utils as mycolors
from esp_lab.utils.netcdf_utils import atomic_to_netcdf

import rtenso_case as rc

MONTH_NAMES = {2: "Feb", 5: "May", 8: "Aug", 11: "Nov"}


def to_datetime(times):
    return [datetime(t.year, t.month, 15) for t in np.atleast_1d(times)]

In [ ]:
MAP_STYLE = {
    # field: (contour interval, min, max, colormap)
    "TS": (0.25, -3.0, 3.0, "blue2red"),
    "TREFHT": (0.25, -3.0, 3.0, "blue2red"),
    "PRECT": (0.5, -5.0, 5.0, "BrBG"),
    "PSL": (0.5, -6.0, 6.0, "blue2red"),
    "U850": (0.5, -6.0, 6.0, "blue2red"),
}


def map_cmap(name, ci, cmin, cmax):
    if name == "blue2red":
        cmap = mycolors.blue2red_cmap(np.arange(cmin, cmax + ci, ci))
    else:
        cmap = plt.get_cmap(name)
    return cmap


def draw_map(fig, da, nrow, ncol, index, title, field, scale=1.0):
    # Robinson map in the ESP-Lab style; NaN (e.g. land for SST) is drawn light grey.
    ci, cmin, cmax, cmap = MAP_STYLE[field]
    ci, cmin, cmax = ci * scale, cmin * scale, cmax * scale
    levels = np.arange(cmin, cmax + ci / 2, ci)
    cmap = map_cmap(cmap, ci, cmin, cmax)
    norm = BoundaryNorm(levels, ncolors=cmap.N)
    values, lon = add_cyclic_point(da.values, coord=da["lon"].values)
    values = np.ma.filled(values, np.nan)  # cartopy wraps pcolormesh only with transparent missing values
    ax = fig.add_subplot(nrow, ncol, index, projection=ccrs.Robinson(central_longitude=180))
    ax.set_facecolor("0.85")  # missing values (e.g. land for SST) show as light grey
    cntr = ax.pcolormesh(lon, da["lat"].values, values, cmap=cmap, norm=norm,
                         shading="nearest", rasterized=True, transform=ccrs.PlateCarree())
    ax.coastlines(lw=0.5)
    ax.set_global()
    ax.set_title(title, fontsize=10)
    return ax, cntr

### Run controls

In [ ]:
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"         # Feb/Aug drift correction: "lead_drift" or "target_month"

FIELD = "TS"                       # TS, TREFHT, PRECT, PSL
CASE_KEYS = list(rc.CASES)
INIT_MONTHS = [2, 5, 8, 11]
FOCUS = {"label": "DJF 2015/16", "months": [(2015, 12), (2016, 1), (2016, 2)]}
ACC_REGIONS = ["Global", "Tropics", "TropPacific", "NHextratropics", "NorthAmerica"]

OUTDIR = rc.diag_dir("RTENSO2026", "case_study", "pattern_skill", rc.FIELDS[FIELD]["realm"], FIELD)
OUTDIR.mkdir(parents=True, exist_ok=True)

### Load anomalies

In [ ]:
%%time
obs = rc.obs_anomaly(FIELD, mode=derivation_mode)

anoms = {}
for case in CASE_KEYS:
    for init in INIT_MONTHS:
        if init not in rc.CASES[case]["init_months"]:
            continue
        anom = rc.hindcast_anomaly(rc.CASES[case]["case_prefix"], init, FIELD, method=CLIM_METHOD, mode=derivation_mode)
        if anom is None:
            print(f"[{case} init{init:02d}] not available")
            continue
        obs_matched = rc.match_obs(anom, obs)
        if FIELD == "TS":  # verify SST: ice-free ocean (HadISST2 ocean points within 60S-60N)
            obs_matched = obs_matched.where(abs(obs_matched["lat"]) <= 60.0)
            anom = anom.where(obs_matched.notnull())
        anoms[(case, init)] = (anom, obs_matched)
        print(f"[{case} init{init:02d}] {anom.sizes['member']} members; {anom.attrs['climatology_method']}")

### Focus-season anomaly maps

Ensemble-mean anomalies over the focus season from every start, with the global pattern ACC in each title.

In [ ]:
ncol = 1 + len(CASE_KEYS)
fig = plt.figure(figsize=(5.2 * ncol, 3.3 * len(INIT_MONTHS)))
obs_focus = rc.obs_target_mean(obs, FOCUS["months"])
if FIELD == "TS":
    obs_focus = obs_focus.where(abs(obs_focus["lat"]) <= 60.0)
ax, cntr = draw_map(fig, obs_focus, len(INIT_MONTHS), ncol, 1, f"Observed {FOCUS['label']}", FIELD)
for row, init in enumerate(INIT_MONTHS):
    for col, case in enumerate(CASE_KEYS, start=1):
        if (case, init) not in anoms:
            continue
        anom, _ = anoms[(case, init)]
        focus = rc.target_mean(anom.mean("member"), FOCUS["months"])
        if focus is None:
            continue
        acc = float(rc.pattern_acc(focus, obs_focus, "Global"))
        lead = [i for i, t in enumerate(anom["time"].values) if (t.year, t.month) == FOCUS["months"][1]][0]
        draw_map(fig, focus, len(INIT_MONTHS), ncol, row * ncol + col + 1,
                 f"{case}, {MONTH_NAMES[init]} start (lead {lead})\nACC={acc:.2f}", FIELD)
cbar = fig.colorbar(cntr, ax=fig.axes, orientation="horizontal", fraction=0.025, pad=0.04, shrink=0.6)
cbar.set_label(f"{FIELD} anomaly ({rc.FIELDS[FIELD]['units']})")
fig.suptitle(f"{rc.FIELDS[FIELD]['long_name']} anomalies, {FOCUS['label']}", fontsize=14)
print(rc.save_figure(fig, "rtenso2026", "anomaly_map", FIELD, FOCUS["label"].replace("/", "")))
plt.show()

### Pattern ACC and RMSE by lead season

Non-overlapping 3-month lead seasons (season 1 = leads 0-2), as in the 1a/1b seasonal products.

In [ ]:
%%time
metrics = {}
for (case, init), (anom, obs_matched) in anoms.items():
    fc_seas = rc.seasonal_leads(anom)
    ob_seas = rc.seasonal_leads(obs_matched)
    ens_seas = fc_seas.mean("member")
    metrics[(case, init)] = xr.Dataset({
        "acc_ens": xr.concat([rc.pattern_acc(ens_seas, ob_seas, r) for r in ACC_REGIONS], dim="region"),
        "acc_member": xr.concat([rc.pattern_acc(fc_seas, ob_seas, r) for r in ACC_REGIONS], dim="region"),
        "rmse_ens": xr.concat([rc.pattern_rmse(ens_seas, ob_seas, r) for r in ACC_REGIONS], dim="region"),
        "rmse_member": xr.concat([rc.pattern_rmse(fc_seas, ob_seas, r) for r in ACC_REGIONS], dim="region"),
    }).assign_coords(region=ACC_REGIONS).load()
    source = rc.CASES[case]["source"]
    atomic_to_netcdf(metrics[(case, init)].drop_vars("time"),
                     OUTDIR / f"{source}_init{init:02d}_{FIELD}_pattern_skill_y{rc.CASE_YEAR}_clim_{rc.CLIM_YEARS[0]}_{rc.CLIM_YEARS[1]}_s8.nc")

In [ ]:
def plot_metric(name, ylabel, ylim=None):
    fig, axes = plt.subplots(len(ACC_REGIONS), len(INIT_MONTHS), figsize=(4.2 * len(INIT_MONTHS), 2.6 * len(ACC_REGIONS)),
                             sharex="col", sharey="row" if ylim is None else True, squeeze=False)
    for col, init in enumerate(INIT_MONTHS):
        for row, region in enumerate(ACC_REGIONS):
            ax = axes[row, col]
            for case in CASE_KEYS:
                ds = metrics.get((case, init))
                if ds is None:
                    continue
                color = rc.CASES[case]["color"]
                season = ds["season"].values
                members = ds[f"{name}_member"].sel(region=region)
                ax.fill_between(season, members.min("member"), members.max("member"), color=color, alpha=0.2)
                ax.plot(season, ds[f"{name}_ens"].sel(region=region), color=color, marker="o", lw=2, label=case)
            if ylim is not None:
                ax.set_ylim(*ylim)
            ax.axhline(0.0, color="0.5", lw=0.8)
            ax.grid(alpha=0.3)
            if col == 0:
                ax.set_ylabel(f"{region}\n{ylabel}")
            if row == 0:
                ax.set_title(f"{MONTH_NAMES[init]} {rc.CASE_YEAR} start")
            if row == len(ACC_REGIONS) - 1:
                ds = next(metrics[(c, init)] for c in CASE_KEYS if (c, init) in metrics)
                ax.set_xticks(ds["season"].values)
                ax.set_xticklabels([rc.target_season_label(t) for t in ds["time"].values], rotation=60, fontsize=8)
    handles = {l: h for ax in axes.flat for h, l in zip(*ax.get_legend_handles_labels())}
    fig.legend(handles.values(), handles.keys(), loc="upper center", ncol=len(handles), fontsize=10, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"{FIELD} pattern {name.upper()} by lead season (line: ensemble mean, shading: member range)", fontsize=13, y=1.05)
    fig.tight_layout()
    return fig


fig = plot_metric("acc", "ACC", ylim=(-1.0, 1.0))
print(rc.save_figure(fig, "rtenso2026", "pattern_acc", FIELD))
plt.show()
fig = plot_metric("rmse", f"RMSE ({rc.FIELDS[FIELD]['units']})")
print(rc.save_figure(fig, "rtenso2026", "pattern_rmse", FIELD))
plt.show()

In [ ]:
table = []
for (case, init), ds in metrics.items():
    for s, t in zip(ds["season"].values, ds["time"].values):
        table.append({"case": case, "start": MONTH_NAMES[init], "season": int(s), "target": rc.target_season_label(t),
                      **{f"ACC_{r}": round(float(ds["acc_ens"].sel(region=r, season=s)), 3) for r in ACC_REGIONS}})
acc_table = pd.DataFrame(table)
acc_table.to_csv(OUTDIR / f"RTENSO2026_case_{FIELD}_pattern_acc.csv", index=False)
acc_table.pivot_table(index=["start", "season", "target"], columns="case", values="ACC_Global")